# ContinuityGuard: catch continuity drift in AI-written microdrama scripts
**Aditya Raj · Kuku AI Generalist assignment · Problem 3**

Before any video is generated, ContinuityGuard checks a whole series for **the same name spelled differently,
one character under several speaker labels, and contradicting appearance** (eye or hair colour, scars, signature items).
Every flag gives the episode, timestamp, an exact quote cut from the script by code, a severity, and a ready-made fix.

**This notebook is saved with its outputs. You can read it top to bottom without running anything.**
The saved outputs come from the bundled 3-episode **demo show**, run in code-only mode. This public copy leaves out the
confidential script pack. The notebook with saved outputs from the full 50-episode pack is in the submission Drive folder
(`02_Prototype/ContinuityGuard_with_outputs.ipynb`).

**To run it yourself:** `Runtime → Run all`. You don't need to upload anything; with no files it checks the bundled demo show.
To check something else, put files in `/content/inputs/` (drag them into the Colab file panel, or tick `UPLOAD_FILES` in step 2):
- a show as a `.zip` of episode `.txt` files
- optionally `raw_outputs.zip`, to replay recorded model answers without a key
- optionally `golden_set.json` and `decisions_pack.yaml`, for the evaluation and the patched scripts

For live AI, add a key in Colab Secrets (`GEMINI_API_KEY`), or tick `ASK_FOR_KEY` in step 3. To swap the model, change
`MODEL` and `BASE_URL`; any OpenAI-compatible endpoint works.

| Step | What it does |
|---|---|
| 1 | Gets the code from GitHub and installs 3 small packages |
| 2 | Finds the inputs: the show `.zip` if present (otherwise the demo show), recorded answers, golden set and decisions |
| 3 | Picks the mode: live AI (it tests which models your key can reach and keeps backups), recorded answers, or code-only |
| 4 | Runs the check and shows the issues, the High flags and the full report |
| 5 | Shows the hand-offs: series bible, prompt block and patched-script diff |
| 6 | Runs the evaluation: golden set, planted errors, one-prompt baseline and clean control |
| 7 | Downloads every output as one zip |


In [1]:
#@title 1. Get the code
REPO_URL = "https://github.com/TheDeveloperAAA/continuity-guard"  #@param {type:"string"}
import os, sys, glob, json, shutil, subprocess, zipfile
BASE = "/content"
os.makedirs(f"{BASE}/inputs", exist_ok=True)
CODE = f"{BASE}/continuity-guard"
if not os.path.exists(CODE):
    if subprocess.run(["git", "clone", "-q", REPO_URL, CODE]).returncode != 0:
        sys.exit("Could not clone the repo; check REPO_URL.")
os.chdir(CODE)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "--root-user-action=ignore", "-r", "requirements.txt"], check=True)
sys.path.insert(0, CODE)
import cguard
from cguard.llm import load_recorded_answers
import pandas as pd
from IPython.display import HTML, Markdown, display
pd.set_option("display.max_colwidth", 120)
CACHE = f"{BASE}/.cg_cache"                    # saved model answers: reruns take seconds
ENV = {**os.environ, "PYTHONHASHSEED": "0"}    # fixed item order, so reruns are reproducible

def run(args):
    """Run run.py and stream its log."""
    p = subprocess.Popen([sys.executable, "run.py"] + args, env=ENV,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    return p.wait()

print("ContinuityGuard", cguard.__version__, "ready at", CODE)


ContinuityGuard 1.7 ready at /content/continuity-guard


In [2]:
#@title 2. Inputs: a show .zip if present, otherwise the bundled demo show
UPLOAD_FILES = False  #@param {type:"boolean"}
if UPLOAD_FILES:
    from google.colab import files
    for name, data in files.upload().items():
        open(f"{BASE}/inputs/{os.path.basename(name)}", "wb").write(data)
INPUTS = sorted(glob.glob(f"{BASE}/inputs/*"))
def pick(test):
    return next((p for p in INPUTS if test(os.path.basename(p))), "")
RECORDED = pick(lambda n: n == "raw_outputs.zip")                       # recorded answers of the final live run
PACK = pick(lambda n: n.endswith(".zip") and not n.startswith("raw_outputs"))
DECISIONS = pick(lambda n: n.startswith("decisions") and n.endswith((".yaml", ".yml")))
GOLDEN = pick(lambda n: n.startswith("golden") and n.endswith(".json"))
if PACK:
    SHOW = f"{BASE}/work/show"
    shutil.rmtree(SHOW, ignore_errors=True)
    zipfile.ZipFile(PACK).extractall(SHOW)
    subdirs = [d for d in glob.glob(f"{SHOW}/*") if os.path.isdir(d)]
    SHOW = subdirs[0] if len(subdirs) == 1 else SHOW
else:
    SHOW = os.path.abspath("samples/demo_show")
episodes = sorted(glob.glob(f"{SHOW}/**/*.txt", recursive=True))
print("Show:", os.path.basename(SHOW), "|", len(episodes), "episode files")
print("Recorded answers:", os.path.basename(RECORDED) or "none", "| decisions:", os.path.basename(DECISIONS) or "none",
      "| golden set:", os.path.basename(GOLDEN) or "none")


Show: demo_show | 3 episode files
Recorded answers: none | decisions: none | golden set: none


In [3]:
#@title 3. Mode: live AI, recorded answers, or code-only
MODEL = "gemini-3.6-flash"  #@param {type:"string"}
BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai"  #@param {type:"string"}
ASK_FOR_KEY = False  #@param {type:"boolean"}
key = os.environ.get("GEMINI_API_KEY", "") or os.environ.get("LLM_API_KEY", "")
try:
    from google.colab import userdata
    for name in ("GEMINI_API_KEY", "LLM_API_KEY"):
        try:
            key = key or (userdata.get(name) or "").strip()
        except Exception:
            pass
except Exception:
    pass
if not key and ASK_FOR_KEY:
    import getpass
    key = getpass.getpass("Paste an API key: ").strip()
MODE, FALLBACKS, HOW = "code", [], "code-only (no key and no recorded answers)"
if key:
    # test which models this key can reach right now; keep the first that answers plus up to 3 backups
    ENV["LLM_API_KEY"] = key
    candidates = [MODEL]
    if "generativelanguage" in BASE_URL:
        candidates += ["gemini-flash-latest", "gemini-3.6-flash", "gemini-3.5-flash", "gemini-3-flash-preview", "gemini-flash-lite-latest"]
    run(["smoke", "--candidates", ",".join(dict.fromkeys(candidates)), "--base-url", BASE_URL, "--save", f"{BASE}/working_models.json"])
    working = json.load(open(f"{BASE}/working_models.json"))["working"]
    if working:
        MODEL, FALLBACKS = working[0], working[1:4]
        MODE, HOW = "full", f"live AI with {MODEL}" + (f" (backups: {', '.join(FALLBACKS)})" if FALLBACKS else "")
    else:
        HOW = "code-only (no model answered with this key right now)"
if MODE == "code" and RECORDED:
    n = load_recorded_answers(RECORDED, CACHE, "gemini-3.6-flash")
    MODEL, FALLBACKS = "gemini-3.6-flash", []
    ENV["LLM_API_KEY"] = "replay-no-key"
    MODE, HOW = "full", f"AI steps replay the {n} recorded answers in {os.path.basename(RECORDED)} (no key, no cost)"
print("Mode:", HOW)


Mode: code-only (no key and no recorded answers)


## 4. Run the check
Pipeline: parse → code checks (spelling variants, speaker labels, colour, scar and item patterns) → **AI Reader**
(10 episodes per request) → candidate merge → **AI Judge** (same or different? mistake or deliberate?) → **code guards** → flags.
These guards are why you can trust the output:
- Quotes are always cut from the script by code, never by the model.
- Citations that don't match the script are rejected.
- The AI can't quietly delete a code finding; a disagreement goes to *needs review*.
- Names used together in one line can't be merged.
- Dissimilar merges go to a human.

In [4]:
#@title 4a. Run
OUT = f"{BASE}/outputs/check"
args = ["check", "--input", SHOW, "--out", OUT, "--mode", MODE, "--model", MODEL, "--fallbacks", ",".join(FALLBACKS),
        "--base-url", BASE_URL, "--cache", CACHE]
if DECISIONS:
    args += ["--decisions", DECISIONS]
rc = run(args)
FINAL = "06" if os.path.exists(f"{OUT}/06_final_flags.csv") else "05"
print("\nExit code", rc, "| outputs:", sorted(os.listdir(OUT)))


{
 "before_human": {
  "flags_total": 5,
  "flag": 3,
  "needs_review": 2,
  "info": 0,
  "high": 2,
  "medium": 1,
  "low": 0,
  "issue_groups": 4,
  "spoken_mentions": 3
 },
 "after_human": null
}
Report: /content/outputs/check/05_ai_report.html

Exit code 0 | outputs: ['00_run_manifest.json', '01_parse_summary.json', '03_candidates.json', '04_judge_decisions.json', '05_ai_flags.csv', '05_ai_report.html', '05_ai_report.md', '07_prompt_block.txt', '07_series_bible.json', '07_series_bible.md']


In [5]:
#@title 4b. Summary and the issues to decide (one decision fixes every mention in its group)
report_md = open(glob.glob(f"{OUT}/{FINAL}_*report.md")[0], encoding="utf-8").read()
display(Markdown(report_md.split("## Issues to decide")[0]))
flags = pd.read_csv(glob.glob(f"{OUT}/{FINAL}_*flags.csv")[0])
groups = (flags.groupby(["group_id", "type", "canonical", "status"])
               .agg(instead_of=("found", lambda s: ", ".join(sorted(set(map(str, s))))),
                    episodes=("episode", lambda s: ",".join(map(str, sorted(set(s))))),
                    mentions=("flag_id", "count"),
                    high=("severity", lambda s: int((s == "High").sum())),
                    source=("source", "first"))
               .reset_index().sort_values(["status", "high", "mentions"], ascending=[True, False, False]))
display(groups[groups.status.isin(["flag", "needs_review"])].reset_index(drop=True))


# ContinuityGuard report: demo_show (before human decisions)

Model: none (code only) | Mode: code | Episodes: 3 | Blocks: 38 | Run: 2026-11-07 17:29 UTC

## Summary

- **4 issue groups to decide** (one decision fixes every mention in its group)
- Flags: **2 High** (spoken), 1 Medium (camera/action/labels), 0 Low (notes) | 2 need review | 0 info rows (deliberate aliases to map)
- Mentions inside spoken dialogue: 3



,group_id,type,canonical,status,instead_of,episodes,mentions,high,source
0,G002,NAME,Raichand,flag,"Raichandani, Raychand","2,3",2,1,code
1,G001,NAME,Vikram,flag,Vikrant,3,1,1,code
2,G003,SPEAKER,MEERA,needs_review,MS. MEERA,3,1,0,code
3,T01,TRAIT,hazel,needs_review,dark brown eyes,3,1,0,code


In [6]:
#@title 4c. High-severity flags (spoken dialogue): exact quote, timestamp and fix
high = flags[(flags.severity == "High") & (flags.status == "flag")]
display(high[["flag_id", "episode", "where", "speaker", "found", "canonical", "snippet", "fix", "source"]].reset_index(drop=True))


,flag_id,episode,where,speaker,found,canonical,snippet,fix,source
0,F0002,3,00:17.60,ARJUN,Raichandani,Raichand,"""The Raichandani Group cancelled your contract an hour ago, Vikram.""","replace ""Raichandani"" with ""Raichand""",code
1,F0001,3,00:21.10,VIKRAM,Vikrant,Vikram,"""Who gave you the authority, Vikrant?""","replace ""Vikrant"" with ""Vikram""",code


In [7]:
#@title 4d. The full report (the same file as in the outputs folder)
display(HTML(open(glob.glob(f"{OUT}/{FINAL}_*report.html")[0], encoding="utf-8").read()))


## 5. Hand-offs to the rest of the pipeline

In [8]:
#@title 5. Series bible, prompt block for the next writing run, and patched scripts
for name in ["07_series_bible.md", "07_prompt_block.txt"]:
    p = f"{OUT}/{name}"
    if os.path.exists(p):
        text = open(p, encoding="utf-8").read()
        display(Markdown(f"### {name}\n\n" + (text if name.endswith(".md") else "```\n" + text + "\n```")))
diff = f"{OUT}/08_fixes.diff"
if os.path.exists(diff):
    lines = open(diff, encoding="utf-8").read().splitlines()
    print(f"08_fixes.diff: {sum(l.startswith('+') and not l.startswith('+++') for l in lines)} changed lines "
          f"across {len(os.listdir(f'{OUT}/08_fixed_scripts'))} patched episode files. First 40 lines:\n")
    print("\n".join(lines[:40]))
else:
    print("No patched scripts (they are written only when a decisions file is given).")


### 07_series_bible.md

# Series bible (continuity reference)

## Names to use

- **Vikram** (never: Vikrant)
- **Raichand** (never: Raichandani, Raychand)

## Characters

| Character | Speaker labels | Lines | Episodes | Traits |
|---|---|---|---|---|
| Vikram | VIKRAM (6) | 6 | 1-3 |  |
| Meera | MEERA (4) | 4 | 1-2 |  |
| Arjun | ARJUN (4) | 4 | 2-3 | eye_colour: hazel |
| Tanya | TANYA (2) | 2 | 1,3 |  |
| Mysterious Rider | MYSTERIOUS RIDER (2) | 2 | 1 |  |
| Ms. Meera | MS. MEERA (1) | 1 | 3 |  |

### 07_prompt_block.txt

```
SERIES CONTINUITY SHEET - use these exact names and traits; do not invent variants.
- Always write "Vikram" (never: Vikrant).
- Always write "Raichand" (never: Raichandani, Raychand).
- Arjun: eye colour: hazel.

```

No patched scripts (they are written only when a decisions file is given).


## 6. Evaluation
The suite runs these checks:
- **Golden set:** real issues in the show, checked by hand.
- **Frozen test set:** errors planted from the show's own text.
- **Tuning set:** planted errors used while building.
- **One-prompt AI baseline:** the whole show in a single request.
- **Clean control:** a script with no errors, to count false alarms.

Relationship flips are planted on purpose to show what the tool does **not** cover yet.

In [9]:
#@title 6. Run the evaluation suite
EVAL = f"{BASE}/outputs/evaluation"
args = ["evaluate", "--input", SHOW, "--out", EVAL, "--mode", MODE, "--model", MODEL, "--fallbacks", ",".join(FALLBACKS),
        "--base-url", BASE_URL, "--cache", CACHE, "--clean", os.path.abspath("samples/demo_show_clean")]
if GOLDEN: args += ["--golden", GOLDEN]
if DECISIONS: args += ["--decisions", DECISIONS]
rc = run(args)
display(Markdown(open(f"{EVAL}/00_evaluation_summary.md", encoding="utf-8").read()))


# Evaluation summary

| Step | Status |
|---|---|
| main | ok |
| code | ok |
| test | ok |
| baseline | skipped (no model in code-only mode) |
| dev | ok |
| clean | ok |

## planted test (10 planted errors)

| Method | In-scope catch rate |
|---|---|
| Full pipeline (code + AI) | 8/9 = 89% |
| Code only | 8/9 = 89% |

| Error type | Full | Code only |
|---|---|---|
| PERSON_NAME | 4/4 | 4/4 |
| SPEAKER_LABEL | 3/4 | 3/4 |
| COLOUR | 1/1 | 1/1 |
| RELATIONSHIP_OUT_OF_SCOPE | 0/1 | 0/1 |

## planted dev (5 planted errors)

| Method | In-scope catch rate |
|---|---|
| Full pipeline (code + AI) | 3/5 = 60% |
| Code only | 3/5 = 60% |

| Error type | Full | Code only |
|---|---|---|
| PERSON_NAME | 1/2 | 1/2 |
| SPEAKER_LABEL | 1/2 | 1/2 |
| COLOUR | 1/1 | 1/1 |

**Clean control (no errors planted):** full mode raised 0 High / 0 flags; code only 0 High / 0 flags.



# Evaluation summary

| Step | Status |
|---|---|
| main | ok |
| code | ok |
| test | ok |
| baseline | skipped (no model in code-only mode) |
| dev | ok |
| clean | ok |

## planted test (10 planted errors)

| Method | In-scope catch rate |
|---|---|
| Full pipeline (code + AI) | 8/9 = 89% |
| Code only | 8/9 = 89% |

| Error type | Full | Code only |
|---|---|---|
| PERSON_NAME | 4/4 | 4/4 |
| SPEAKER_LABEL | 3/4 | 3/4 |
| COLOUR | 1/1 | 1/1 |
| RELATIONSHIP_OUT_OF_SCOPE | 0/1 | 0/1 |

## planted dev (5 planted errors)

| Method | In-scope catch rate |
|---|---|
| Full pipeline (code + AI) | 3/5 = 60% |
| Code only | 3/5 = 60% |

| Error type | Full | Code only |
|---|---|---|
| PERSON_NAME | 1/2 | 1/2 |
| SPEAKER_LABEL | 1/2 | 1/2 |
| COLOUR | 1/1 | 1/1 |

**Clean control (no errors planted):** full mode raised 0 High / 0 flags; code only 0 High / 0 flags.


**Cost and time of the live run** (`raw_outputs/check/00_run_manifest.json` and `llm_calls/`):
- **Per 50-episode show:** 7 requests, about 166k input and 94k output tokens. Of the output, 67k are thinking tokens, which Gemini bills as output.
- **Time:** about 7.5 minutes of model time on the free tier.
- **Cost:** about $0.48 per show at Gemini 3.6 Flash list prices ($0.75 in / $3.75 out per million tokens, until 31 Dec 2026).
  At the 2027 prices it would be about $0.96. Batch pricing halves both.
- **Editor time:** about 16 decisions, each fixing every mention in its group.

A replay of recorded answers costs nothing and uses 0 tokens.

In [10]:
#@title 7. Download every output as one zip (Colab)
shutil.make_archive(f"{BASE}/continuityguard_outputs", "zip", f"{BASE}/outputs")
try:
    from google.colab import files
    files.download(f"{BASE}/continuityguard_outputs.zip")
except Exception:
    print("Saved", f"{BASE}/continuityguard_outputs.zip")


Saved /content/continuityguard_outputs.zip
